# Experiment B — Στατιστική ενίσχυση (N boost)

**Τι ελέγχουμε:** τίποτα νέο — στενεύουμε τα CIs στα αποφασιστικά cells και δίνουμε στο Qwen3-4B το N που χρειάζεται για σημαντικότητα (τώρα Δ=+0.94, p=0.06 με N=3).

**Σχεδιασμός:** star topology (όπως το πρώτο campaign), +5 fresh runs στα no_comm/baseline/counterfactual/framing_competitive ανά μοντέλο. Τα νέα runs μπαίνουν σε ξεχωριστό φάκελο (`results_nboost`) και συγχωνεύονται στην ανάλυση ως πρόσθετη ανεξάρτητη παρτίδα.

## Setup — install, GPU check, clone, HF token

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (χρειάζεται για Llama/Gemma)

**Προσοχή:** το repo πρέπει να έχει γίνει push με τις αλλαγές του Phase 1.5 (topologies + `--action-retries`) πριν τρέξει αυτό το notebook.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo
# sanity: Phase-1.5 features present
assert 'clique' in open('topology.py').read(), 'Repo lacks Phase-1.5 topologies — push first!'
!ls

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print('No HF_TOKEN secret (fine for Qwen):', e)

In [ ]:
MODEL_ID = 'Qwen/Qwen3-4B'
# Τρέξε για όλα: Qwen/Qwen2.5-7B-Instruct, google/gemma-2-9b-it,
# meta-llama/Llama-3.1-8B-Instruct, google/gemma-2-2b-it, Qwen/Qwen3-4B

In [ ]:
!python run_all_scenarios.py --provider local --model-id $MODEL_ID \
    --scenarios baseline counterfactual framing_competitive \
    --n-runs 5 --max-new-tokens 256 \
    --out-dir-base /kaggle/working/results_nboost \
    --zip-after-each --zip-mirror /kaggle/working

In [ ]:
# Μόνο για Qwen3-4B: συμπλήρωση των ablations από N=3 σε N=5 (2 έξτρα runs)
# !python run_all_scenarios.py --provider local --model-id Qwen/Qwen3-4B \
#     --scenarios silence no_sense \
#     --n-runs 2 --max-new-tokens 256 \
#     --out-dir-base /kaggle/working/results_nboost --zip-after-each --zip-mirror /kaggle/working